# Codificación de elementos textuales

## Importación de librerias necesarias

In [1]:
import os
os.getenv('LD_LIBRARY_PATH')

'/usr/lib/x86_64-linux-gnu'

In [2]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

2025-10-25 15:41:01.128247: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-25 15:41:01.140883: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-25 15:41:01.144441: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-25 15:41:01.155151: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-25 15:41:03.105524: W tensorflow/compiler/tf2

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


I0000 00:00:1761424867.051916  386083 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1761424867.343977  386083 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1761424867.344170  386083 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355


In [3]:
%load_ext IPython.extensions.autoreload
%autoreload 2

In [4]:
import os
from pathlib import Path
import sys

def find_src_folder(current_path: Path, folder_name: str = 'src') -> Path:
    search_directories = [current_path] + list(current_path.parents)
    for parent in search_directories:
        if parent.name == folder_name:
            return parent.parent
    return current_path

src_path = find_src_folder(Path.cwd(), 'src')
sys.path.append(str(src_path))

print(f"JAVA_HOME: {os.environ.get('JAVA_HOME')}")
print(f"TFHUB_CACHE_DIR: {os.environ.get('TFHUB_CACHE_DIR')}")

JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64
TFHUB_CACHE_DIR: /mnt/d/Maestría/Amazon Reviews Code/tf_cache


In [ ]:
import logging
it is very
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

In [6]:
import pandas as pd
import tensorflow_hub as hub

In [7]:
from src.utils.spark import SparkUtils

In [8]:
!nvidia-smi

Sat Oct 25 15:41:35 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.274.02             Driver Version: 535.274.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3050        Off | 00000000:10:00.0  On |                  N/A |
|  0%   51C    P2              21W /  70W |    720MiB /  6144MiB |     16%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [9]:
spark_utils = SparkUtils('encoding')
spark = spark_utils.spark

2025-10-25 15:41:58,893 - SparkCreator - INFO - Starts creating environment


:: loading settings :: url = jar:file:/mnt/d/Maestr%c3%ada/Amazon%20Reviews%20Code/.venv-linux/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/edgar/.ivy2/cache
The jars for the packages stored in: /home/edgar/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a29533ba-e959-4986-b428-eef16acf1e17;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 113ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   

In [10]:
REGENERATE_INTERMEDIATE_TABLES = False

In [11]:
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window

## Importar información transformada

In [12]:
items_category_encoded = spark.read.format('delta').load(spark_utils.path(
    'main_category_encoded', catalog = 'silver.preprocess'
))

In [13]:
items_category_encoded.show(10)

+--------------------+--------------------+--------+-----------+--------------+-------------+-----+-----+-----------+--------------------+--------------------+------+--------------------+--------------------+--------------------+---------------------+--------------------+--------------------+------------+-----------------------+-----------------------------+--------------------------------+-------------------------+-----------------------------------+-------------------------------------+-----------------------------+-----------------------------+--------------------------+
|               title|       main_category|features|description|average_rating|rating_number|price|store|parent_asin|          categories|             details|images|description_colapsed|   features_colapsed|       colapsed_text|colapsed_text_spelled| colapsed_text_words|colapsed_text_length|review_count|main_category_computers|main_category_all_electronics|main_category_home_audio_theater|main_category_amazon_home|

In [14]:
if REGENERATE_INTERMEDIATE_TABLES:
    main_category_encoded_split = (
        items_category_encoded
            .select(
                F.col('parent_asin'),
                F.explode(F.split(
                    'colapsed_text',
                    '\\.'
                )).alias('colapsed_text_split')
            )
            .withColumn(
                'colapsed_text_split',
                F.trim(F.col('colapsed_text_split'))
            )
            .withColumn(
                'record_id', F.row_number().over(
                    Window.orderBy(
                        F.col('parent_asin').asc(),
                        F.col('colapsed_text_split').asc()
                    )
                )
            )
    )

    (
        main_category_encoded_split.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path('main_category_encoded_split', catalog = 'silver.models'))
    )
    
main_category_encoded_split = spark.read.format('delta').load(spark_utils.path(
    'main_category_encoded_split', catalog = 'silver.models'
))

In [15]:
main_category_encoded_split.filter(F.col('parent_asin') == '0594514843').show(1)

+-----------+--------------------+---------+
|parent_asin| colapsed_text_split|record_id|
+-----------+--------------------+---------+
| 0594514843|Custom clip mecha...|      201|
+-----------+--------------------+---------+
only showing top 1 row



## Codificar textos generados en proceso de limpieza de datos

In [16]:
@F.pandas_udf(returnType=T.ArrayType(T.FloatType()))
def encode_texts_split(texts: pd.Series) -> pd.Series:
    model = hub.load(MODULE_URL)
    text_list = [str(text) if pd.notna(text) and text != "" else "" for text in texts]
    embeddings = model(text_list).numpy()
    return pd.Series([embedding.tolist() for embedding in embeddings])

main_category_encoded_split_with_embeddings = main_category_encoded_split.withColumn(
    "text_embeddings", 
    encode_texts_split(F.col("colapsed_text_split"))
)


In [17]:
main_category_encoded_split.select('parent_asin').distinct().count()

694836

In [16]:
main_category_encoded_split_sample = (
    main_category_encoded_split
        .withColumn('id2', F.dense_rank().over(
            Window.orderBy('parent_asin')
        ))
        .filter(F.col('id2') <= 100_000)
        .drop('id2')
)

In [16]:
if REGENERATE_INTERMEDIATE_TABLES:
    (
        main_category_encoded_split_sample.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path('main_category_encoded_split_sample', catalog='silver.models'))
    )

main_category_encoded_split_sample = spark.read.format('delta').load(spark_utils.path(
    'main_category_encoded_split_sample', catalog = 'silver.models'
))
main_category_encoded_split_sample.count()

2643697

In [15]:
meta_items_texts = spark.read.format('delta').load(spark_utils.path(
    'meta_items_texts',
    catalog = 'silver.preprocess'
))

In [ ]:
meta_items = spark.read.format('delta').load(spark_utils.path(
    'meta_items', catalog = 'silver.preprocess'
))

In [ ]:
main_category_encoded_split_sample.limit(10.toPandas())

+-----------+--------------------+---------+
|parent_asin| colapsed_text_split|record_id|
+-----------+--------------------+---------+
| 0110400550|100% Brand New, h...|        1|
| 0110400550|Color: Pink and W...|        2|
| 0110400550|Pink & White 3D M...|        3|
| 0110400550|Precise openings ...|        4|
| 0511189877|Al clickrs are br...|        5|
| 0511189877|Brand new clickr ...|        6|
| 0511189877|If you have any q...|        7|
| 0511189877|Please be advised...|        8|
| 0511189877|That doesn't in a...|        9|
| 0511189877|URC CLIKR-5 Time ...|       10|
+-----------+--------------------+---------+
only showing top 10 rows



In [ ]:
import tensorflow_hub as hub
import numpy as np
from pyspark.sql import Window, functions as F

model = hub.load(MODULE_URL)

def embed_batch(texts, batch_size=1024):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        emb = model(batch).numpy()
        embeddings.append(emb)
    return np.vstack(embeddings)

def process_to_parquet(df, col: str, id_col: str, batch_size: int, tf_batch: int, parquet_path: str):
    w = Window.orderBy(id_col)
    df_idx = df.withColumn("rn", F.row_number().over(w))
    count = df.count()
    print("====Parquets to process=====", count)
    offset = 0
    batch_idx = 0
    while offset < count:
        print("====Processing batch=====", batch_idx, "offset", offset)
        batch_df = (
            df_idx
            .filter((F.col("rn") > offset) & (F.col("rn") <= offset + batch_size))
            .select(id_col, col)
        )
        pdf = batch_df.toPandas()
        texts = pdf[col].fillna("").astype(str).tolist()
        embs = embed_batch(texts, batch_size=tf_batch)
        emb_df = spark.createDataFrame(
            list(zip(pdf[id_col].tolist(), embs.tolist())),
            schema=[id_col, "text_embeddings"]
        )
        mode = "overwrite" if batch_idx == 0 else "append"
        emb_df.write.mode(mode).parquet(parquet_path)
        offset += batch_size
        batch_idx += 1

process_to_parquet(
    main_category_encoded_split_sample,
    col="colapsed_text_split",
    id_col="record_id",
    batch_size=2500,
    tf_batch=256,
    parquet_path=spark_utils.path('main_category_encoded_split_with_embeddings_sample_2', catalog='silver.models')
)


In [15]:
main_category_encoded_split_with_embeddings_sample = spark.read.format('parquet').load(spark_utils.path(
    'main_category_encoded_split_with_embeddings_sample_3', catalog = 'silver.models'
))

In [16]:
main_category_encoded_split_with_embeddings_sample.count()

1315000

## Promediar embeddings por producto

In [22]:
encoded_split_with_embeddings_parent_asin = (
    main_category_encoded_split_with_embeddings_sample.alias('A')
        .join(
            main_category_encoded_split.alias('B'),
            on = 'record_id',
            how = 'inner'
        )
        .select(
            'B.parent_asin',
            'A.text_embeddings'
        )
)

In [30]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, FloatType
from pyspark.sql.functions import col
from pyspark.sql.pandas.functions import PandasUDFType
import pandas as pd
import numpy as np

spark = SparkSession.builder.getOrCreate()

spark_df = encoded_split_with_embeddings_parent_asin

schema = StructType([
    StructField("parent_asin", StringType()),
    StructField("avg_embedding", ArrayType(FloatType()))
])

@pandas_udf(schema, functionType=PandasUDFType.GROUPED_MAP)
def average_embeddings_udf(pdf: pd.DataFrame) -> pd.DataFrame:
    avg_emb = np.mean(np.stack(pdf["text_embeddings"].to_numpy()), axis=0).astype(float)
    return pd.DataFrame({"parent_asin": [pdf["parent_asin"].iloc[0]], "avg_embedding": [avg_emb.tolist()]})

avg_encoded_split_with_embeddings = spark_df.groupby("parent_asin").apply(average_embeddings_udf)


In [ ]:
if REGENERATE_INTERMEDIATE_TABLES:
    (
        avg_encoded_split_with_embeddings.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path('avg_encoded_split_with_embeddings', catalog='silver.models'))
    )

avg_encoded_split_with_embeddings = spark.read.format('delta').load(spark_utils.path(
    'avg_encoded_split_with_embeddings', catalog = 'silver.models'
))

## Codificar informaciòn de reseñas

In [33]:
reviews_fixed_rating = spark.read.format('delta').load(spark_utils.path(
    'reviews_fixed_rating', catalog = 'silver.preprocess'
))

In [34]:
reviews_fixed_rating.show(1)

+------+---------+--------------------+-------------+------------+-----------+------+--------------------+--------------+
|rating|    title|                text|    timestamp|helpful_vote|parent_asin|images|        text_unified|rating_boolean|
+------+---------+--------------------+-------------+------------+-----------+------+--------------------+--------------+
|   4.0|It works!|Works as describe...|1620148192836|           0| B07PW7PSY9|    []|It works!. Works ...|             1|
+------+---------+--------------------+-------------+------------+-----------+------+--------------------+--------------+
only showing top 1 row



Seleccionar reseñas de productos previamente encontrados

In [ ]:
reviews_fixed_rating_related = reviews_fixed_rating.alias('A').join(
    avg_encoded_split_with_embeddings.alias('B'),
    on = 'parent_asin',
    how = 'inner'
).withColumn(
    'record_id', F.row_number().over(
        Window.orderBy(
            F.col('parent_asin').asc(),
            F.col('text').asc()
        )
    )
)


In [51]:
if REGENERATE_INTERMEDIATE_TABLES:
    (
        reviews_fixed_rating_related.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path('reviews_fixed_rating_related', catalog='silver.models'))
    )

reviews_fixed_rating_related = spark.read.format('delta').load(spark_utils.path(
    'reviews_fixed_rating_related', catalog = 'silver.models'
))

In [46]:
if REGENERATE_INTERMEDIATE_TABLES:
    reviews_fixed_rating_related_split = (
        reviews_fixed_rating_related
            .select(
                F.col('record_id'),
                F.explode(F.split(
                    'text_unified',
                    '\\.'
                )).alias('text_split')
            )
            .withColumn(
                'text_split',
                F.trim(F.col('text_split'))
            )
            .withColumn(
                'record_phrase_id', F.row_number().over(
                    Window.orderBy(
                        F.col('record_id').asc(),
                        F.col('text_split').asc()
                    )
                )
            )
            .filter(
                F.col('text_split').isNotNull() & (F.col('text_split') != '') &
                (F.length(F.col('text_split')) > 0)
            )
    )

    (
        reviews_fixed_rating_related_split.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path('reviews_fixed_rating_related_split', catalog = 'silver.models'))
    )
    
reviews_fixed_rating_related_split = spark.read.format('delta').load(spark_utils.path(
    'reviews_fixed_rating_related_split', catalog = 'silver.models'
))

In [47]:
reviews_fixed_rating_related_split.count()

13833018

In [48]:
reviews_fixed_rating_related_embeddings = spark.read.format('parquet').load(spark_utils.path(
    'reviews_fixed_rating_related_embeddings', catalog = 'silver.models'
))
reviews_fixed_rating_related_embeddings.count()
reviews_fixed_rating_related_embeddings.show(1)
reviews_fixed_rating_related_embeddings.printSchema()

+----------------+--------------------+
|record_phrase_id|     text_embeddings|
+----------------+--------------------+
|          113314|[-0.0151835065335...|
+----------------+--------------------+
only showing top 1 row

root
 |-- record_phrase_id: long (nullable = true)
 |-- text_embeddings: array (nullable = true)
 |    |-- element: double (containsNull = true)



In [54]:
reviews_fixed_rating_related_embeddings.alias('A').join(
    reviews_fixed_rating_related_split.alias('B'),
    on = 'record_phrase_id',
    how = 'inner'
).join(
    reviews_fixed_rating_related.alias('C'),
    on = 'record_id',
    how = 'inner'
).select(
    'C.parent_asin',
).show(100)

+-----------+
|parent_asin|
+-----------+
| 0528881469|
| 0528881469|
| 0528881469|
| 0899332897|
| 0899332897|
| 0899332897|
| 0899332897|
| 0899332897|
| 0899332897|
| 0899332897|
| 0899332897|
| 0899332897|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972683275|
| 0972

In [28]:
import tensorflow_hub as hub
import numpy as np
from pyspark.sql import Window, functions as F

model = hub.load(MODULE_URL)

def embed_batch(texts, batch_size=1024):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        emb = model(batch).numpy()
        embeddings.append(emb)
    return np.vstack(embeddings)

def process_to_parquet(df, col: str, id_col: str, batch_size: int, tf_batch: int, parquet_path: str):
    w = Window.orderBy(id_col)
    df_idx = df.withColumn("rn", F.row_number().over(w))
    count = df.count()
    print("====Parquets to process=====", count)
    offset = 0
    batch_idx = 0
    while offset < count:
        print("====Processing batch=====", batch_idx, "offset", offset)
        batch_df = (
            df_idx
            .filter((F.col("rn") > offset) & (F.col("rn") <= offset + batch_size))
            .select(id_col, col)
        )
        pdf = batch_df.toPandas()
        texts = pdf[col].fillna("").astype(str).tolist()
        embs = embed_batch(texts, batch_size=tf_batch)
        emb_df = spark.createDataFrame(
            list(zip(pdf[id_col].tolist(), embs.tolist())),
            schema=[id_col, "text_embeddings"]
        )
        mode = "overwrite" if batch_idx == 0 else "append"
        emb_df.write.mode(mode).parquet(parquet_path)
        offset += batch_size
        batch_idx += 1

process_to_parquet(
    reviews_fixed_rating_related_split,
    col="text_split",
    id_col="record_phrase_id",
    batch_size=2500,
    tf_batch=256,
    parquet_path=spark_utils.path('reviews_fixed_rating_related_embeddings', catalog='silver.models')
)


2025-09-28 17:44:37,265 - absl - INFO - Fingerprint not found. Saved model loading will continue.
2025-09-28 17:44:37,265 - absl - INFO - path_and_singleprint metric could not be logged. Saved model loading will continue.


====Parquets to process===== 13833018
====Processing batch===== 0 offset 0


====Processing batch===== 1 offset 2500


====Processing batch===== 2 offset 5000


====Processing batch===== 3 offset 7500


...

====Processing batch===== 51 offset 127500


====Processing batch===== 52 offset 130000


====Processing batch===== 53 offset 132500


[1734.565s][warning][gc,alloc] Executor task launch worker for task 0.0 in stage 401.0 (TID 4045): Retried waiting for GCLocker too often allocating 131074 words
18:04:27.265 [Executor task launch worker for task 0.0 in stage 401.0 (TID 4045)] ERROR org.apache.spark.executor.Executor - Exception in task 0.0 in stage 401.0 (TID 4045)
java.lang.OutOfMemoryError: Java heap space
18:04:27.285 [Executor task launch worker for task 0.0 in stage 401.0 (TID 4045)] ERROR org.apache.spark.util.SparkUncaughtExceptionHandler - Uncaught exception in thread Thread[Executor task launch worker for task 0.0 in stage 401.0 (TID 4045),5,main]
java.lang.OutOfMemoryError: Java heap space
18:04:27.289 [task-result-getter-2] ERROR org.apache.spark.scheduler.TaskSetManager - Task 0 in stage 401.0 failed 1 times; aborting job


Py4JJavaError: An error occurred while calling o2019.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 401.0 failed 1 times, most recent failure: Lost task 0.0 in stage 401.0 (TID 4045) (192.168.1.33 executor driver): java.lang.OutOfMemoryError: Java heap space

Driver stacktrace:
	at org.apache.spark.scheduler.DAGScheduler.failJobAndIndependentStages(DAGScheduler.scala:2844)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2(DAGScheduler.scala:2780)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2$adapted(DAGScheduler.scala:2779)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.foreach$(ResizableArray.scala:55)
	at scala.collection.mutable.ArrayBuffer.foreach(ArrayBuffer.scala:49)
	at org.apache.spark.scheduler.DAGScheduler.abortStage(DAGScheduler.scala:2779)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$

## Otros

In [ ]:
items_category_encoded.count()

694836

In [ ]:
items_category_encoded.head(1)

[Row(title='Barnes & Noble FITS 6" TABLET / E-READER NOOK Simple Touch Oliver Cover RED', main_category='home_audio_theater', features=None, description=None, average_rating=None, rating_number=9, price=None, store=None, parent_asin='0594514843', categories=['Electronics', 'eBook Readers & Accessories'], details='{"Product Dimensions":"5.51 x 7.01 x 1 inches","Item Weight":"7.8 ounces","Manufacturer":"Barnes and Noble","Item model number":"9780594514848","Is Discontinued By Manufacturer":"No","Date First Available":"July 15, 2015","Brand":"Barnes & Noble","Color":"Red","Compatible Devices":"E-Readers","Form Factor":"Case","Material":"Faux Leather"}', images=None, description_colapsed='The Oliver Cover in Red for NOOK Simple Touch is designed to protect and safeguard your eBook reader. The Nook simple touch case has a smooth, comfortable synthetic leather exterior. Your Nook is held in place with a simple bar-clip, eliminating the need for straps. This Nook case provides easy access to 

In [ ]:
from pyspark.sql.functions import pandas_udf
import pyspark.sql.functions as F

In [ ]:
os.environ["TFHUB_CACHE_DIR"] = "/tmp/tfhub"

In [ ]:
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import UniversalSentenceEncoder
from pyspark.ml import Pipeline
import sparknlp

spark = sparknlp.start(gpu = True)

:: loading settings :: url = jar:file:/mnt/d/Maestr%c3%ada/Amazon%20Reviews%20Code/.venv-linux/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/edgar/.ivy2/cache
The jars for the packages stored in: /home/edgar/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp-gpu_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1778281b-113b-4459-9678-261e38e70eb0;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp-gpu_2.12;5.1.4 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.828 in central
	found com.github.universal-automata#liblevenshtein;3.0.0 in central
	found com.google.protobuf#protobuf-java-util;3.0.0-beta-3 in central
	found com.google.protobuf#protobuf-java;3.0.0-beta-3 in central
	found com.google.code.gson#gson;2.3 in central
	found it.unimi.dsi#fastutil;7.0.12 in central
	found org.projectlombok#lombok;1.16.8 in central
	found com.google.cloud#google-cloud-storage;2.20.1 in central
	found com.google.guava#guava;31.1-jre in central
	found com.google.guava#

In [ ]:


df = spark.createDataFrame([("This is a test",), ("Spark NLP works with USE",)], ["text"])

doc = DocumentAssembler().setInputCol("text").setOutputCol("document")
use = UniversalSentenceEncoder.pretrained("tfhub_use", "en") \
    .setInputCols(["document"]).setOutputCol("sentence_embeddings")

pipeline = Pipeline(stages=[doc, use])
model = pipeline.fit(df)
result = model.transform(df)

result.select("sentence_embeddings.embeddings").show(truncate=False)


tfhub_use download started this may take some time.
Approximate size to download 923.7 MB
[ / ]tfhub_use download started this may take some time.
Approximate size to download 923.7 MB
[ \ ]Download done! Loading the resource.
[ — ]
An error occurred while calling z:com.johnsnowlabs.nlp.pretrained.PythonResourceDownloader.downloadModel.
: java.lang.UnsatisfiedLinkError: no jnitensorflow in java.library.path: [/usr/lib/x86_64-linux-gnu, /usr/java/packages/lib, /usr/lib/x86_64-linux-gnu/jni, /lib/x86_64-linux-gnu, /usr/lib/x86_64-linux-gnu, /usr/lib/jni, /lib, /usr/lib]
	at java.base/java.lang.ClassLoader.loadLibrary(ClassLoader.java:2678)
	at java.base/java.lang.Runtime.loadLibrary0(Runtime.java:830)
	at java.base/java.lang.System.loadLibrary(System.java:1890)
	at org.bytedeco.javacpp.Loader.loadLibrary(Loader.java:1832)
	at org.bytedeco.javacpp.Loader.load(Loader.java:1423)
	at org.bytedeco.javacpp.Loader.load(Loader.java:1234)
	at org.bytedeco.javacpp.Loader.load(Loader.java:1210)
	at

Py4JJavaError: An error occurred while calling z:com.johnsnowlabs.nlp.pretrained.PythonResourceDownloader.downloadModel.
: java.lang.UnsatisfiedLinkError: no jnitensorflow in java.library.path: [/usr/lib/x86_64-linux-gnu, /usr/java/packages/lib, /usr/lib/x86_64-linux-gnu/jni, /lib/x86_64-linux-gnu, /usr/lib/x86_64-linux-gnu, /usr/lib/jni, /lib, /usr/lib]
	at java.base/java.lang.ClassLoader.loadLibrary(ClassLoader.java:2678)
	at java.base/java.lang.Runtime.loadLibrary0(Runtime.java:830)
	at java.base/java.lang.System.loadLibrary(System.java:1890)
	at org.bytedeco.javacpp.Loader.loadLibrary(Loader.java:1832)
	at org.bytedeco.javacpp.Loader.load(Loader.java:1423)
	at org.bytedeco.javacpp.Loader.load(Loader.java:1234)
	at org.bytedeco.javacpp.Loader.load(Loader.java:1210)
	at org.tensorflow.internal.c_api.global.tensorflow.<clinit>(tensorflow.java:12)
	at java.base/java.lang.Class.forName0(Native Method)
	at java.base/java.lang.Class.forName(Class.java:398)
	at org.bytedeco.javacpp.Loader.

In [ ]:
import tensorflow
import tempfile
import tensorflow.keras.models


In [ ]:
model(
    [
        'How are you today?', 'What time is it now?', 'Where do you live?', 'Do you like music?', 'Can you help me?', 
        'This is my friend.', 'I have a cat.', 'Do you want some tea?', 'She is very kind.', 'I love reading books.', 
        'It is raining outside.', 'The sun is bright.', 'I am learning Python.', 'Do you play football?', 'They are at school.', 
        'He is watching TV.', 'We are going shopping.', 'I need some water.', 'Do you speak English?', 'She likes painting.', 
        'The car is fast.', 'Please close the door.', 'This is very easy.', 'He is my brother.', 'We are good friends.', 
        'I want to travel soon.', 'What is your favorite color?', 'Do you have a pen?', 'She is drinking coffee.', 'They are playing chess.', 
        'My house is small.', 'I am very happy.', 'Do you know him?', 'We like pizza.', 'It is very cold.', 
        'The train is late.', 'I am reading a book.', 'He likes to swim.', 'She is at the park.', 'They are working hard.', 
        'I want to learn more.', 'Please open the window.', 'It is very hot today.', 'We are going to the beach.', 'Do you like movies?', 
        'She is listening to music.', 'He is cooking dinner.', 'I have a new phone.', 'They are my parents.', 'We are eating lunch.', 
        'Do you like coffee?', 'She is reading a story.', 'He is riding a bike.', 'We are playing games.', 'I like chocolate.', 
        'The sky is blue.', 'It is snowing today.', 'She is writing a letter.', 'Do you want some water?', 'He is sleeping now.', 
        'We are singing together.', 'The dog is barking.', 'I love ice cream.', 'She is talking to him.', 'They are watching a movie.', 
        'I am learning English.', 'Do you like apples?', 'He is running fast.', 'We are waiting here.', 'It is very dark.', 
        'The bus is coming.', 'She is smiling.', 'I am feeling tired.', 'They are cooking rice.', 'We are having fun.', 
        'He is playing guitar.', 'Do you like cats?', 'She is reading a magazine.', 'It is very late.', 'We are going home.', 
        'I need your help.', 'The flowers are beautiful.', 'She is dancing.', 'He is drinking juice.', 'They are my classmates.', 
        'We are studying math.', 'Do you like ice cream?', 'I am writing a note.', 'She is waiting outside.', 'He is fixing the car.', 
        'The birds are singing.', 'It is a sunny day.', 'I am going to sleep.', 'We are watching TV.', 'She is drawing a picture.', 
        'He is walking home.', 'Do you like football?', 'They are playing outside.', 'I am cooking pasta.', 'The baby is crying.', 
        'She is making a cake.', 'He is studying now.', 'We are talking together.', 'It is very quiet.', 'I am feeling hungry.', 
        'They are laughing loudly.', 'The river is long.', 'She is reading the news.', 'He is opening the box.', 'We are packing bags.',
        'How are you today?', 'What time is it now?', 'Where do you live?', 'Do you like music?', 'Can you help me?', 
        'This is my friend.', 'I have a cat.', 'Do you want some tea?', 'She is very kind.', 'I love reading books.', 
        'It is raining outside.', 'The sun is bright.', 'I am learning Python.', 'Do you play football?', 'They are at school.', 
        'He is watching TV.', 'We are going shopping.', 'I need some water.', 'Do you speak English?', 'She likes painting.', 
        'The car is fast.', 'Please close the door.', 'This is very easy.', 'He is my brother.', 'We are good friends.', 
        'I want to travel soon.', 'What is your favorite color?', 'Do you have a pen?', 'She is drinking coffee.', 'They are playing chess.', 
        'My house is small.', 'I am very happy.', 'Do you know him?', 'We like pizza.', 'It is very cold.', 
        'The train is late.', 'I am reading a book.', 'He likes to swim.', 'She is at the park.', 'They are working hard.', 
        'I want to learn more.', 'Please open the window.', 'It is very hot today.', 'We are going to the beach.', 'Do you like movies?', 
        'She is listening to music.', 'He is cooking dinner.', 'I have a new phone.', 'They are my parents.', 'We are eating lunch.', 
        'Do you like coffee?', 'She is reading a story.', 'He is riding a bike.', 'We are playing games.', 'I like chocolate.', 
        'The sky is blue.', 'It is snowing today.', 'She is writing a letter.', 'Do you want some water?', 'He is sleeping now.', 
        'We are singing together.', 'The dog is barking.', 'I love ice cream.', 'She is talking to him.', 'They are watching a movie.', 
        'I am learning English.', 'Do you like apples?', 'He is running fast.', 'We are waiting here.', 'It is very dark.', 
        'The bus is coming.', 'She is smiling.', 'I am feeling tired.', 'They are cooking rice.', 'We are having fun.', 
        'He is playing guitar.', 'Do you like cats?', 'She is reading a magazine.', 'It is very late.', 'We are going home.', 
        'I need your help.', 'The flowers are beautiful.', 'She is dancing.', 'He is drinking juice.', 'They are my classmates.', 
        'We are studying math.', 'Do you like ice cream?', 'I am writing a note.', 'She is waiting outside.', 'He is fixing the car.', 
        'The birds are singing.', 'It is a sunny day.', 'I am going to sleep.', 'We are watching TV.', 'She is drawing a picture.', 
        'He is walking home.', 'Do you like football?', 'They are playing outside.', 'I am cooking pasta.', 'The baby is crying.', 
        'She is making a cake.', 'He is studying now.', 'We are talking together.', 'It is very quiet.', 'I am feeling hungry.', 
        'They are laughing loudly.', 'The river is long.', 'She is reading the news.', 'He is opening the box.', 'We are packing bags.',
        'How are you today?', 'What time is it now?', 'Where do you live?', 'Do you like music?', 'Can you help me?', 
        'This is my friend.', 'I have a cat.', 'Do you want some tea?', 'She is very kind.', 'I love reading books.', 
        'It is raining outside.', 'The sun is bright.', 'I am learning Python.', 'Do you play football?', 'They are at school.', 
        'He is watching TV.', 'We are going shopping.', 'I need some water.', 'Do you speak English?', 'She likes painting.', 
        'The car is fast.', 'Please close the door.', 'This is very easy.', 'He is my brother.', 'We are good friends.', 
        'I want to travel soon.', 'What is your favorite color?', 'Do you have a pen?', 'She is drinking coffee.', 'They are playing chess.', 
        'My house is small.', 'I am very happy.', 'Do you know him?', 'We like pizza.', 'It is very cold.', 
        'The train is late.', 'I am reading a book.', 'He likes to swim.', 'She is at the park.', 'They are working hard.', 
        'I want to learn more.', 'Please open the window.', 'It is very hot today.', 'We are going to the beach.', 'Do you like movies?', 
        'She is listening to music.', 'He is cooking dinner.', 'I have a new phone.', 'They are my parents.', 'We are eating lunch.', 
        'Do you like coffee?', 'She is reading a story.', 'He is riding a bike.', 'We are playing games.', 'I like chocolate.', 
        'The sky is blue.', 'It is snowing today.', 'She is writing a letter.', 'Do you want some water?', 'He is sleeping now.', 
        'We are singing together.', 'The dog is barking.', 'I love ice cream.', 'She is talking to him.', 'They are watching a movie.', 
        'I am learning English.', 'Do you like apples?', 'He is running fast.', 'We are waiting here.', 'It is very dark.', 
        'The bus is coming.', 'She is smiling.', 'I am feeling tired.', 'They are cooking rice.', 'We are having fun.', 
        'He is playing guitar.', 'Do you like cats?', 'She is reading a magazine.', 'It is very late.', 'We are going home.', 
        'I need your help.', 'The flowers are beautiful.', 'She is dancing.', 'He is drinking juice.', 'They are my classmates.', 
        'We are studying math.', 'Do you like ice cream?', 'I am writing a note.', 'She is waiting outside.', 'He is fixing the car.', 
        'The birds are singing.', 'It is a sunny day.', 'I am going to sleep.', 'We are watching TV.', 'She is drawing a picture.', 
        'He is walking home.', 'Do you like football?', 'They are playing outside.', 'I am cooking pasta.', 'The baby is crying.', 
        'She is making a cake.', 'He is studying now.', 'We are talking together.', 'It is very quiet.', 'I am feeling hungry.', 
        'They are laughing loudly.', 'The river is long.', 'She is reading the news.', 'He is opening the box.', 'We are packing bags.',
        'How are you today?', 'What time is it now?', 'Where do you live?', 'Do you like music?', 'Can you help me?', 
        'This is my friend.', 'I have a cat.', 'Do you want some tea?', 'She is very kind.', 'I love reading books.', 
        'It is raining outside.', 'The sun is bright.', 'I am learning Python.', 'Do you play football?', 'They are at school.', 
        'He is watching TV.', 'We are going shopping.', 'I need some water.', 'Do you speak English?', 'She likes painting.', 
        'The car is fast.', 'Please close the door.', 'This is very easy.', 'He is my brother.', 'We are good friends.', 
        'I want to travel soon.', 'What is your favorite color?', 'Do you have a pen?', 'She is drinking coffee.', 'They are playing chess.', 
        'My house is small.', 'I am very happy.', 'Do you know him?', 'We like pizza.', 'It is very cold.', 
        'The train is late.', 'I am reading a book.', 'He likes to swim.', 'She is at the park.', 'They are working hard.', 
        'I want to learn more.', 'Please open the window.', 'It is very hot today.', 'We are going to the beach.', 'Do you like movies?', 
        'She is listening to music.', 'He is cooking dinner.', 'I have a new phone.', 'They are my parents.', 'We are eating lunch.', 
        'Do you like coffee?', 'She is reading a story.', 'He is riding a bike.', 'We are playing games.', 'I like chocolate.', 
        'The sky is blue.', 'It is snowing today.', 'She is writing a letter.', 'Do you want some water?', 'He is sleeping now.', 
        'We are singing together.', 'The dog is barking.', 'I love ice cream.', 'She is talking to him.', 'They are watching a movie.', 
        'I am learning English.', 'Do you like apples?', 'He is running fast.', 'We are waiting here.', 'It is very dark.', 
        'The bus is coming.', 'She is smiling.', 'I am feeling tired.', 'They are cooking rice.', 'We are having fun.', 
        'He is playing guitar.', 'Do you like cats?', 'She is reading a magazine.', 'It is very late.', 'We are going home.', 
        'I need your help.', 'The flowers are beautiful.', 'She is dancing.', 'He is drinking juice.', 'They are my classmates.', 
        'We are studying math.', 'Do you like ice cream?', 'I am writing a note.', 'She is waiting outside.', 'He is fixing the car.', 
        'The birds are singing.', 'It is a sunny day.', 'I am going to sleep.', 'We are watching TV.', 'She is drawing a picture.', 
        'He is walking home.', 'Do you like football?', 'They are playing outside.', 'I am cooking pasta.', 'The baby is crying.', 
        'She is making a cake.', 'He is studying now.', 'We are talking together.', 'It is very quiet.', 'I am feeling hungry.', 
        'They are laughing loudly.', 'The river is long.', 'She is reading the news.', 'He is opening the box.', 'We are packing bags.',
        'How are you today?', 'What time is it now?', 'Where do you live?', 'Do you like music?', 'Can you help me?', 
        'This is my friend.', 'I have a cat.', 'Do you want some tea?', 'She is very kind.', 'I love reading books.', 
        'It is raining outside.', 'The sun is bright.', 'I am learning Python.', 'Do you play football?', 'They are at school.', 
        'He is watching TV.', 'We are going shopping.', 'I need some water.', 'Do you speak English?', 'She likes painting.', 
        'The car is fast.', 'Please close the door.', 'This is very easy.', 'He is my brother.', 'We are good friends.', 
        'I want to travel soon.', 'What is your favorite color?', 'Do you have a pen?', 'She is drinking coffee.', 'They are playing chess.', 
        'My house is small.', 'I am very happy.', 'Do you know him?', 'We like pizza.', 'It is very cold.', 
        'The train is late.', 'I am reading a book.', 'He likes to swim.', 'She is at the park.', 'They are working hard.', 
        'I want to learn more.', 'Please open the window.', 'It is very hot today.', 'We are going to the beach.', 'Do you like movies?', 
        'She is listening to music.', 'He is cooking dinner.', 'I have a new phone.', 'They are my parents.', 'We are eating lunch.', 
        'Do you like coffee?', 'She is reading a story.', 'He is riding a bike.', 'We are playing games.', 'I like chocolate.', 
        'The sky is blue.', 'It is snowing today.', 'She is writing a letter.', 'Do you want some water?', 'He is sleeping now.', 
        'We are singing together.', 'The dog is barking.', 'I love ice cream.', 'She is talking to him.', 'They are watching a movie.', 
        'I am learning English.', 'Do you like apples?', 'He is running fast.', 'We are waiting here.', 'It is very dark.', 
        'The bus is coming.', 'She is smiling.', 'I am feeling tired.', 'They are cooking rice.', 'We are having fun.', 
        'He is playing guitar.', 'Do you like cats?', 'She is reading a magazine.', 'It is very late.', 'We are going home.', 
        'I need your help.', 'The flowers are beautiful.', 'She is dancing.', 'He is drinking juice.', 'They are my classmates.', 
        'We are studying math.', 'Do you like ice cream?', 'I am writing a note.', 'She is waiting outside.', 'He is fixing the car.', 
        'The birds are singing.', 'It is a sunny day.', 'I am going to sleep.', 'We are watching TV.', 'She is drawing a picture.', 
        'He is walking home.', 'Do you like football?', 'They are playing outside.', 'I am cooking pasta.', 'The baby is crying.', 
        'She is making a cake.', 'He is studying now.', 'We are talking together.', 'It is very quiet.', 'I am feeling hungry.', 
        'They are laughing loudly.', 'The river is long.', 'She is reading the news.', 'He is opening the box.', 'We are packing bags.',
        'How are you today?', 'What time is it now?', 'Where do you live?', 'Do you like music?', 'Can you help me?', 
        'This is my friend.', 'I have a cat.', 'Do you want some tea?', 'She is very kind.', 'I love reading books.', 
        'It is raining outside.', 'The sun is bright.', 'I am learning Python.', 'Do you play football?', 'They are at school.', 
        'He is watching TV.', 'We are going shopping.', 'I need some water.', 'Do you speak English?', 'She likes painting.', 
        'The car is fast.', 'Please close the door.', 'This is very easy.', 'He is my brother.', 'We are good friends.', 
        'I want to travel soon.', 'What is your favorite color?', 'Do you have a pen?', 'She is drinking coffee.', 'They are playing chess.', 
        'My house is small.', 'I am very happy.', 'Do you know him?', 'We like pizza.', 'It is very cold.', 
        'The train is late.', 'I am reading a book.', 'He likes to swim.', 'She is at the park.', 'They are working hard.', 
        'I want to learn more.', 'Please open the window.', 'It is very hot today.', 'We are going to the beach.', 'Do you like movies?', 
        'She is listening to music.', 'He is cooking dinner.', 'I have a new phone.', 'They are my parents.', 'We are eating lunch.', 
        'Do you like coffee?', 'She is reading a story.', 'He is riding a bike.', 'We are playing games.', 'I like chocolate.', 
        'The sky is blue.', 'It is snowing today.', 'She is writing a letter.', 'Do you want some water?', 'He is sleeping now.', 
        'We are singing together.', 'The dog is barking.', 'I love ice cream.', 'She is talking to him.', 'They are watching a movie.', 
        'I am learning English.', 'Do you like apples?', 'He is running fast.', 'We are waiting here.', 'It is very dark.', 
        'The bus is coming.', 'She is smiling.', 'I am feeling tired.', 'They are cooking rice.', 'We are having fun.', 
        'He is playing guitar.', 'Do you like cats?', 'She is reading a magazine.', 'It is very late.', 'We are going home.', 
        'I need your help.', 'The flowers are beautiful.', 'She is dancing.', 'He is drinking juice.', 'They are my classmates.', 
        'We are studying math.', 'Do you like ice cream?', 'I am writing a note.', 'She is waiting outside.', 'He is fixing the car.', 
        'The birds are singing.', 'It is a sunny day.', 'I am going to sleep.', 'We are watching TV.', 'She is drawing a picture.', 
        'He is walking home.', 'Do you like football?', 'They are playing outside.', 'I am cooking pasta.', 'The baby is crying.', 
        'She is making a cake.', 'He is studying now.', 'We are talking together.', 'It is very quiet.', 'I am feeling hungry.', 
        'They are laughing loudly.', 'The river is long.', 'She is reading the news.', 'He is opening the box.', 'We are packing bags.',
        'How are you today?', 'What time is it now?', 'Where do you live?', 'Do you like music?', 'Can you help me?', 
        'This is my friend.', 'I have a cat.', 'Do you want some tea?', 'She is very kind.', 'I love reading books.', 
        'It is raining outside.', 'The sun is bright.', 'I am learning Python.', 'Do you play football?', 'They are at school.', 
        'He is watching TV.', 'We are going shopping.', 'I need some water.', 'Do you speak English?', 'She likes painting.', 
        'The car is fast.', 'Please close the door.', 'This is very easy.', 'He is my brother.', 'We are good friends.', 
        'I want to travel soon.', 'What is your favorite color?', 'Do you have a pen?', 'She is drinking coffee.', 'They are playing chess.', 
        'My house is small.', 'I am very happy.', 'Do you know him?', 'We like pizza.', 'It is very cold.', 
        'The train is late.', 'I am reading a book.', 'He likes to swim.', 'She is at the park.', 'They are working hard.', 
        'I want to learn more.', 'Please open the window.', 'It is very hot today.', 'We are going to the beach.', 'Do you like movies?', 
        'She is listening to music.', 'He is cooking dinner.', 'I have a new phone.', 'They are my parents.', 'We are eating lunch.', 
        'Do you like coffee?', 'She is reading a story.', 'He is riding a bike.', 'We are playing games.', 'I like chocolate.', 
        'The sky is blue.', 'It is snowing today.', 'She is writing a letter.', 'Do you want some water?', 'He is sleeping now.', 
        'We are singing together.', 'The dog is barking.', 'I love ice cream.', 'She is talking to him.', 'They are watching a movie.', 
        'I am learning English.', 'Do you like apples?', 'He is running fast.', 'We are waiting here.', 'It is very dark.', 
        'The bus is coming.', 'She is smiling.', 'I am feeling tired.', 'They are cooking rice.', 'We are having fun.', 
        'He is playing guitar.', 'Do you like cats?', 'She is reading a magazine.', 'It is very late.', 'We are going home.', 
        'I need your help.', 'The flowers are beautiful.', 'She is dancing.', 'He is drinking juice.', 'They are my classmates.', 
        'We are studying math.', 'Do you like ice cream?', 'I am writing a note.', 'She is waiting outside.', 'He is fixing the car.', 
        'The birds are singing.', 'It is a sunny day.', 'I am going to sleep.', 'We are watching TV.', 'She is drawing a picture.', 
        'He is walking home.', 'Do you like football?', 'They are playing outside.', 'I am cooking pasta.', 'The baby is crying.', 
        'She is making a cake.', 'He is studying now.', 'We are talking together.', 'It is very quiet.', 'I am feeling hungry.', 
        'They are laughing loudly.', 'The river is long.', 'She is reading the news.', 'He is opening the box.', 'We are packing bags.',
        'How are you today?', 'What time is it now?', 'Where do you live?', 'Do you like music?', 'Can you help me?', 
        'This is my friend.', 'I have a cat.', 'Do you want some tea?', 'She is very kind.', 'I love reading books.', 
        'It is raining outside.', 'The sun is bright.', 'I am learning Python.', 'Do you play football?', 'They are at school.', 
        'He is watching TV.', 'We are going shopping.', 'I need some water.', 'Do you speak English?', 'She likes painting.', 
        'The car is fast.', 'Please close the door.', 'This is very easy.', 'He is my brother.', 'We are good friends.', 
        'I want to travel soon.', 'What is your favorite color?', 'Do you have a pen?', 'She is drinking coffee.', 'They are playing chess.', 
        'My house is small.', 'I am very happy.', 'Do you know him?', 'We like pizza.', 'It is very cold.', 
        'The train is late.', 'I am reading a book.', 'He likes to swim.', 'She is at the park.', 'They are working hard.', 
        'I want to learn more.', 'Please open the window.', 'It is very hot today.', 'We are going to the beach.', 'Do you like movies?', 
        'She is listening to music.', 'He is cooking dinner.', 'I have a new phone.', 'They are my parents.', 'We are eating lunch.', 
        'Do you like coffee?', 'She is reading a story.', 'He is riding a bike.', 'We are playing games.', 'I like chocolate.', 
        'The sky is blue.', 'It is snowing today.', 'She is writing a letter.', 'Do you want some water?', 'He is sleeping now.', 
        'We are singing together.', 'The dog is barking.', 'I love ice cream.', 'She is talking to him.', 'They are watching a movie.', 
        'I am learning English.', 'Do you like apples?', 'He is running fast.', 'We are waiting here.', 'It is very dark.', 
        'The bus is coming.', 'She is smiling.', 'I am feeling tired.', 'They are cooking rice.', 'We are having fun.', 
        'He is playing guitar.', 'Do you like cats?', 'She is reading a magazine.', 'It is very late.', 'We are going home.', 
        'I need your help.', 'The flowers are beautiful.', 'She is dancing.', 'He is drinking juice.', 'They are my classmates.', 
        'We are studying math.', 'Do you like ice cream?', 'I am writing a note.', 'She is waiting outside.', 'He is fixing the car.', 
        'The birds are singing.', 'It is a sunny day.', 'I am going to sleep.', 'We are watching TV.', 'She is drawing a picture.', 
        'He is walking home.', 'Do you like football?', 'They are playing outside.', 'I am cooking pasta.', 'The baby is crying.', 
        'She is making a cake.', 'He is studying now.', 'We are talking together.', 'It is very quiet.', 'I am feeling hungry.', 
        'They are laughing loudly.', 'The river is long.', 'She is reading the news.', 'He is opening the box.', 'We are packing bags.',
        'How are you today?', 'What time is it now?', 'Where do you live?', 'Do you like music?', 'Can you help me?', 
        'This is my friend.', 'I have a cat.', 'Do you want some tea?', 'She is very kind.', 'I love reading books.', 
        'It is raining outside.', 'The sun is bright.', 'I am learning Python.', 'Do you play football?', 'They are at school.', 
        'He is watching TV.', 'We are going shopping.', 'I need some water.', 'Do you speak English?', 'She likes painting.', 
        'The car is fast.', 'Please close the door.', 'This is very easy.', 'He is my brother.', 'We are good friends.', 
        'I want to travel soon.', 'What is your favorite color?', 'Do you have a pen?', 'She is drinking coffee.', 'They are playing chess.', 
        'My house is small.', 'I am very happy.', 'Do you know him?', 'We like pizza.', 'It is very cold.', 
        'The train is late.', 'I am reading a book.', 'He likes to swim.', 'She is at the park.', 'They are working hard.', 
        'I want to learn more.', 'Please open the window.', 'It is very hot today.', 'We are going to the beach.', 'Do you like movies?', 
        'She is listening to music.', 'He is cooking dinner.', 'I have a new phone.', 'They are my parents.', 'We are eating lunch.', 
        'Do you like coffee?', 'She is reading a story.', 'He is riding a bike.', 'We are playing games.', 'I like chocolate.', 
        'The sky is blue.', 'It is snowing today.', 'She is writing a letter.', 'Do you want some water?', 'He is sleeping now.', 
        'We are singing together.', 'The dog is barking.', 'I love ice cream.', 'She is talking to him.', 'They are watching a movie.', 
        'I am learning English.', 'Do you like apples?', 'He is running fast.', 'We are waiting here.', 'It is very dark.', 
        'The bus is coming.', 'She is smiling.', 'I am feeling tired.', 'They are cooking rice.', 'We are having fun.', 
        'He is playing guitar.', 'Do you like cats?', 'She is reading a magazine.', 'It is very late.', 'We are going home.', 
        'I need your help.', 'The flowers are beautiful.', 'She is dancing.', 'He is drinking juice.', 'They are my classmates.', 
        'We are studying math.', 'Do you like ice cream?', 'I am writing a note.', 'She is waiting outside.', 'He is fixing the car.', 
        'The birds are singing.', 'It is a sunny day.', 'I am going to sleep.', 'We are watching TV.', 'She is drawing a picture.', 
        'He is walking home.', 'Do you like football?', 'They are playing outside.', 'I am cooking pasta.', 'The baby is crying.', 
        'She is making a cake.', 'He is studying now.', 'We are talking together.', 'It is very quiet.', 'I am feeling hungry.', 
        'They are laughing loudly.', 'The river is long.', 'She is reading the news.', 'He is opening the box.', 'We are packing bags.',
        'How are you today?', 'What time is it now?', 'Where do you live?', 'Do you like music?', 'Can you help me?', 
        'This is my friend.', 'I have a cat.', 'Do you want some tea?', 'She is very kind.', 'I love reading books.', 
        'It is raining outside.', 'The sun is bright.', 'I am learning Python.', 'Do you play football?', 'They are at school.', 
        'He is watching TV.', 'We are going shopping.', 'I need some water.', 'Do you speak English?', 'She likes painting.', 
        'The car is fast.', 'Please close the door.', 'This is very easy.', 'He is my brother.', 'We are good friends.', 
        'I want to travel soon.', 'What is your favorite color?', 'Do you have a pen?', 'She is drinking coffee.', 'They are playing chess.', 
        'My house is small.', 'I am very happy.', 'Do you know him?', 'We like pizza.', 'It is very cold.', 
        'The train is late.', 'I am reading a book.', 'He likes to swim.', 'She is at the park.', 'They are working hard.', 
        'I want to learn more.', 'Please open the window.', 'It is very hot today.', 'We are going to the beach.', 'Do you like movies?', 
        'She is listening to music.', 'He is cooking dinner.', 'I have a new phone.', 'They are my parents.', 'We are eating lunch.', 
        'Do you like coffee?', 'She is reading a story.', 'He is riding a bike.', 'We are playing games.', 'I like chocolate.', 
        'The sky is blue.', 'It is snowing today.', 'She is writing a letter.', 'Do you want some water?', 'He is sleeping now.', 
        'We are singing together.', 'The dog is barking.', 'I love ice cream.', 'She is talking to him.', 'They are watching a movie.', 
        'I am learning English.', 'Do you like apples?', 'He is running fast.', 'We are waiting here.', 'It is very dark.', 
        'The bus is coming.', 'She is smiling.', 'I am feeling tired.', 'They are cooking rice.', 'We are having fun.', 
        'He is playing guitar.', 'Do you like cats?', 'She is reading a magazine.', 'It is very late.', 'We are going home.', 
        'I need your help.', 'The flowers are beautiful.', 'She is dancing.', 'He is drinking juice.', 'They are my classmates.', 
        'We are studying math.', 'Do you like ice cream?', 'I am writing a note.', 'She is waiting outside.', 'He is fixing the car.', 
        'The birds are singing.', 'It is a sunny day.', 'I am going to sleep.', 'We are watching TV.', 'She is drawing a picture.', 
        'He is walking home.', 'Do you like football?', 'They are playing outside.', 'I am cooking pasta.', 'The baby is crying.', 
        'She is making a cake.', 'He is studying now.', 'We are talking together.', 'It is very quiet.', 'I am feeling hungry.', 
        'They are laughing loudly.', 'The river is long.', 'She is reading the news.', 'He is opening the box.', 'We are packing bags.',
        'How are you today?', 'What time is it now?', 'Where do you live?', 'Do you like music?', 'Can you help me?', 
        'This is my friend.', 'I have a cat.', 'Do you want some tea?', 'She is very kind.', 'I love reading books.', 
        'It is raining outside.', 'The sun is bright.', 'I am learning Python.', 'Do you play football?', 'They are at school.', 
        'He is watching TV.', 'We are going shopping.', 'I need some water.', 'Do you speak English?', 'She likes painting.', 
        'The car is fast.', 'Please close the door.', 'This is very easy.', 'He is my brother.', 'We are good friends.', 
        'I want to travel soon.', 'What is your favorite color?', 'Do you have a pen?', 'She is drinking coffee.', 'They are playing chess.', 
        'My house is small.', 'I am very happy.', 'Do you know him?', 'We like pizza.', 'It is very cold.', 
        'The train is late.', 'I am reading a book.', 'He likes to swim.', 'She is at the park.', 'They are working hard.', 
        'I want to learn more.', 'Please open the window.', 'It is very hot today.', 'We are going to the beach.', 'Do you like movies?', 
        'She is listening to music.', 'He is cooking dinner.', 'I have a new phone.', 'They are my parents.', 'We are eating lunch.', 
        'Do you like coffee?', 'She is reading a story.', 'He is riding a bike.', 'We are playing games.', 'I like chocolate.', 
        'The sky is blue.', 'It is snowing today.', 'She is writing a letter.', 'Do you want some water?', 'He is sleeping now.', 
        'We are singing together.', 'The dog is barking.', 'I love ice cream.', 'She is talking to him.', 'They are watching a movie.', 
        'I am learning English.', 'Do you like apples?', 'He is running fast.', 'We are waiting here.', 'It is very dark.', 
        'The bus is coming.', 'She is smiling.', 'I am feeling tired.', 'They are cooking rice.', 'We are having fun.', 
        'He is playing guitar.', 'Do you like cats?', 'She is reading a magazine.', 'It is very late.', 'We are going home.', 
        'I need your help.', 'The flowers are beautiful.', 'She is dancing.', 'He is drinking juice.', 'They are my classmates.', 
        'We are studying math.', 'Do you like ice cream?', 'I am writing a note.', 'She is waiting outside.', 'He is fixing the car.', 
        'The birds are singing.', 'It is a sunny day.', 'I am going to sleep.', 'We are watching TV.', 'She is drawing a picture.', 
        'He is walking home.', 'Do you like football?', 'They are playing outside.', 'I am cooking pasta.', 'The baby is crying.', 
        'She is making a cake.', 'He is studying now.', 'We are talking together.', 'It is very quiet.', 'I am feeling hungry.', 
        'They are laughing loudly.', 'The river is long.', 'She is reading the news.', 'He is opening the box.', 'We are packing bags.',
        'How are you today?', 'What time is it now?', 'Where do you live?', 'Do you like music?', 'Can you help me?', 
        'This is my friend.', 'I have a cat.', 'Do you want some tea?', 'She is very kind.', 'I love reading books.', 
        'It is raining outside.', 'The sun is bright.', 'I am learning Python.', 'Do you play football?', 'They are at school.', 
        'He is watching TV.', 'We are going shopping.', 'I need some water.', 'Do you speak English?', 'She likes painting.', 
        'The car is fast.', 'Please close the door.', 'This is very easy.', 'He is my brother.', 'We are good friends.', 
        'I want to travel soon.', 'What is your favorite color?', 'Do you have a pen?', 'She is drinking coffee.', 'They are playing chess.', 
        'My house is small.', 'I am very happy.', 'Do you know him?', 'We like pizza.', 'It is very cold.', 
        'The train is late.', 'I am reading a book.', 'He likes to swim.', 'She is at the park.', 'They are working hard.', 
        'I want to learn more.', 'Please open the window.', 'It is very hot today.', 'We are going to the beach.', 'Do you like movies?', 
        'She is listening to music.', 'He is cooking dinner.', 'I have a new phone.', 'They are my parents.', 'We are eating lunch.', 
        'Do you like coffee?', 'She is reading a story.', 'He is riding a bike.', 'We are playing games.', 'I like chocolate.', 
        'The sky is blue.', 'It is snowing today.', 'She is writing a letter.', 'Do you want some water?', 'He is sleeping now.', 
        'We are singing together.', 'The dog is barking.', 'I love ice cream.', 'She is talking to him.', 'They are watching a movie.', 
        'I am learning English.', 'Do you like apples?', 'He is running fast.', 'We are waiting here.', 'It is very dark.', 
        'The bus is coming.', 'She is smiling.', 'I am feeling tired.', 'They are cooking rice.', 'We are having fun.', 
        'He is playing guitar.', 'Do you like cats?', 'She is reading a magazine.', 'It is very late.', 'We are going home.', 
        'I need your help.', 'The flowers are beautiful.', 'She is dancing.', 'He is drinking juice.', 'They are my classmates.', 
        'We are studying math.', 'Do you like ice cream?', 'I am writing a note.', 'She is waiting outside.', 'He is fixing the car.', 
        'The birds are singing.', 'It is a sunny day.', 'I am going to sleep.', 'We are watching TV.', 'She is drawing a picture.', 
        'He is walking home.', 'Do you like football?', 'They are playing outside.', 'I am cooking pasta.', 'The baby is crying.', 
        'She is making a cake.', 'He is studying now.', 'We are talking together.', 'It is very quiet.', 'I am feeling hungry.', 
        'They are laughing loudly.', 'The river is long.', 'She is reading the news.', 'He is opening the box.', 'We are packing bags.',
        'How are you today?', 'What time is it now?', 'Where do you live?', 'Do you like music?', 'Can you help me?', 
        'This is my friend.', 'I have a cat.', 'Do you want some tea?', 'She is very kind.', 'I love reading books.', 
        'It is raining outside.', 'The sun is bright.', 'I am learning Python.', 'Do you play football?', 'They are at school.', 
        'He is watching TV.', 'We are going shopping.', 'I need some water.', 'Do you speak English?', 'She likes painting.', 
        'The car is fast.', 'Please close the door.', 'This is very easy.', 'He is my brother.', 'We are good friends.', 
        'I want to travel soon.', 'What is your favorite color?', 'Do you have a pen?', 'She is drinking coffee.', 'They are playing chess.', 
        'My house is small.', 'I am very happy.', 'Do you know him?', 'We like pizza.', 'It is very cold.', 
        'The train is late.', 'I am reading a book.', 'He likes to swim.', 'She is at the park.', 'They are working hard.', 
        'I want to learn more.', 'Please open the window.', 'It is very hot today.', 'We are going to the beach.', 'Do you like movies?', 
        'She is listening to music.', 'He is cooking dinner.', 'I have a new phone.', 'They are my parents.', 'We are eating lunch.', 
        'Do you like coffee?', 'She is reading a story.', 'He is riding a bike.', 'We are playing games.', 'I like chocolate.', 
        'The sky is blue.', 'It is snowing today.', 'She is writing a letter.', 'Do you want some water?', 'He is sleeping now.', 
        'We are singing together.', 'The dog is barking.', 'I love ice cream.', 'She is talking to him.', 'They are watching a movie.', 
        'I am learning English.', 'Do you like apples?', 'He is running fast.', 'We are waiting here.', 'It is very dark.', 
        'The bus is coming.', 'She is smiling.', 'I am feeling tired.', 'They are cooking rice.', 'We are having fun.', 
        'He is playing guitar.', 'Do you like cats?', 'She is reading a magazine.', 'It is very late.', 'We are going home.', 
        'I need your help.', 'The flowers are beautiful.', 'She is dancing.', 'He is drinking juice.', 'They are my classmates.', 
        'We are studying math.', 'Do you like ice cream?', 'I am writing a note.', 'She is waiting outside.', 'He is fixing the car.', 
        'The birds are singing.', 'It is a sunny day.', 'I am going to sleep.', 'We are watching TV.', 'She is drawing a picture.', 
        'He is walking home.', 'Do you like football?', 'They are playing outside.', 'I am cooking pasta.', 'The baby is crying.', 
        'She is making a cake.', 'He is studying now.', 'We are talking together.', 'It is very quiet.', 'I am feeling hungry.', 
        'They are laughing loudly.', 'The river is long.', 'She is reading the news.', 'He is opening the box.', 'We are packing bags.',
        'How are you today?', 'What time is it now?', 'Where do you live?', 'Do you like music?', 'Can you help me?', 
        'This is my friend.', 'I have a cat.', 'Do you want some tea?', 'She is very kind.', 'I love reading books.', 
        'It is raining outside.', 'The sun is bright.', 'I am learning Python.', 'Do you play football?', 'They are at school.', 
        'He is watching TV.', 'We are going shopping.', 'I need some water.', 'Do you speak English?', 'She likes painting.', 
        'The car is fast.', 'Please close the door.', 'This is very easy.', 'He is my brother.', 'We are good friends.', 
        'I want to travel soon.', 'What is your favorite color?', 'Do you have a pen?', 'She is drinking coffee.', 'They are playing chess.', 
        'My house is small.', 'I am very happy.', 'Do you know him?', 'We like pizza.', 'It is very cold.', 
        'The train is late.', 'I am reading a book.', 'He likes to swim.', 'She is at the park.', 'They are working hard.', 
        'I want to learn more.', 'Please open the window.', 'It is very hot today.', 'We are going to the beach.', 'Do you like movies?', 
        'She is listening to music.', 'He is cooking dinner.', 'I have a new phone.', 'They are my parents.', 'We are eating lunch.', 
        'Do you like coffee?', 'She is reading a story.', 'He is riding a bike.', 'We are playing games.', 'I like chocolate.', 
        'The sky is blue.', 'It is snowing today.', 'She is writing a letter.', 'Do you want some water?', 'He is sleeping now.', 
        'We are singing together.', 'The dog is barking.', 'I love ice cream.', 'She is talking to him.', 'They are watching a movie.', 
        'I am learning English.', 'Do you like apples?', 'He is running fast.', 'We are waiting here.', 'It is very dark.', 
        'The bus is coming.', 'She is smiling.', 'I am feeling tired.', 'They are cooking rice.', 'We are having fun.', 
        'He is playing guitar.', 'Do you like cats?', 'She is reading a magazine.', 'It is very late.', 'We are going home.', 
        'I need your help.', 'The flowers are beautiful.', 'She is dancing.', 'He is drinking juice.', 'They are my classmates.', 
        'We are studying math.', 'Do you like ice cream?', 'I am writing a note.', 'She is waiting outside.', 'He is fixing the car.', 
        'The birds are singing.', 'It is a sunny day.', 'I am going to sleep.', 'We are watching TV.', 'She is drawing a picture.', 
        'He is walking home.', 'Do you like football?', 'They are playing outside.', 'I am cooking pasta.', 'The baby is crying.', 
        'She is making a cake.', 'He is studying now.', 'We are talking together.', 'It is very quiet.', 'I am feeling hungry.', 
        'They are laughing loudly.', 'The river is long.', 'She is reading the news.', 'He is opening the box.', 'We are packing bags.',
        'How are you today?', 'What time is it now?', 'Where do you live?', 'Do you like music?', 'Can you help me?', 
        'This is my friend.', 'I have a cat.', 'Do you want some tea?', 'She is very kind.', 'I love reading books.', 
        'It is raining outside.', 'The sun is bright.', 'I am learning Python.', 'Do you play football?', 'They are at school.', 
        'He is watching TV.', 'We are going shopping.', 'I need some water.', 'Do you speak English?', 'She likes painting.', 
        'The car is fast.', 'Please close the door.', 'This is very easy.', 'He is my brother.', 'We are good friends.', 
        'I want to travel soon.', 'What is your favorite color?', 'Do you have a pen?', 'She is drinking coffee.', 'They are playing chess.', 
        'My house is small.', 'I am very happy.', 'Do you know him?', 'We like pizza.', 'It is very cold.', 
        'The train is late.', 'I am reading a book.', 'He likes to swim.', 'She is at the park.', 'They are working hard.', 
        'I want to learn more.', 'Please open the window.', 'It is very hot today.', 'We are going to the beach.', 'Do you like movies?', 
        'She is listening to music.', 'He is cooking dinner.', 'I have a new phone.', 'They are my parents.', 'We are eating lunch.', 
        'Do you like coffee?', 'She is reading a story.', 'He is riding a bike.', 'We are playing games.', 'I like chocolate.', 
        'The sky is blue.', 'It is snowing today.', 'She is writing a letter.', 'Do you want some water?', 'He is sleeping now.', 
        'We are singing together.', 'The dog is barking.', 'I love ice cream.', 'She is talking to him.', 'They are watching a movie.', 
        'I am learning English.', 'Do you like apples?', 'He is running fast.', 'We are waiting here.', 'It is very dark.', 
        'The bus is coming.', 'She is smiling.', 'I am feeling tired.', 'They are cooking rice.', 'We are having fun.', 
        'He is playing guitar.', 'Do you like cats?', 'She is reading a magazine.', 'It is very late.', 'We are going home.', 
        'I need your help.', 'The flowers are beautiful.', 'She is dancing.', 'He is drinking juice.', 'They are my classmates.', 
        'We are studying math.', 'Do you like ice cream?', 'I am writing a note.', 'She is waiting outside.', 'He is fixing the car.', 
        'The birds are singing.', 'It is a sunny day.', 'I am going to sleep.', 'We are watching TV.', 'She is drawing a picture.', 
        'He is walking home.', 'Do you like football?', 'They are playing outside.', 'I am cooking pasta.', 'The baby is crying.', 
        'She is making a cake.', 'He is studying now.', 'We are talking together.', 'It is very quiet.', 'I am feeling hungry.', 
        'They are laughing loudly.', 'The river is long.', 'She is reading the news.', 'He is opening the box.', 'We are packing bags.',
    ]
)


<tf.Tensor: shape=(1650, 512), dtype=float32, numpy=
array([[ 0.02284648, -0.01669665,  0.05715492, ..., -0.02046595,
        -0.04852883, -0.02301924],
       [ 0.00399273,  0.03632415,  0.0254656 , ..., -0.00112819,
        -0.05119208,  0.04845312],
       [ 0.00760297,  0.02440583,  0.02337593, ...,  0.03581152,
         0.040789  ,  0.00312181],
       ...,
       [ 0.0185167 ,  0.00373833,  0.01871512, ...,  0.01541435,
        -0.04882324,  0.04261131],
       [ 0.04013611,  0.0466255 ,  0.00799642, ...,  0.05630086,
        -0.0064644 , -0.00677477],
       [ 0.05999969,  0.0819743 ,  0.04052193, ...,  0.06212946,
        -0.01190093, -0.00576386]], shape=(1650, 512), dtype=float32)>